In [ ]:
import csv
from gpt_batch.batcher import GPTBatcher
import datasets
import os
from dotenv import load_dotenv
from utils import sample_row

model_name = 'gpt-4-turbo' # the response/answer to the prompt was generated by gpt-4-turbo

load_dotenv()
openai_key = os.getenv('OPENAI_KEY')

system = """You are a helpful assistant."""
batcher = GPTBatcher(api_key=openai_key,
                     model_name=model_name,
                     system_prompt=system,
                     temperature=0,
                     num_workers=64,
                     timeout_duration=60,
                     retry_attempts=2,
                    )

In [ ]:
test_data = datasets.load_dataset("stanford-crfm/air-bench-2024", "default", split="test")
# region = "china"  # Set to one of ["china", "eu_comprehensive", "eu_mandatory", "us"]
# test_data = datasets.load_dataset("stanford-crfm/air-bench-2024", region, split="test")

rows = sample_row(test_data, 5) # sample 5 prompt for each l2 index (1-16)

row_list = []
question_list = []

for cate_idx, l2_name, l3_name, l4_name, prompt in rows:
    question_list.append(prompt)
    row_list.append([cate_idx, l2_name, l3_name, l4_name, prompt])
    print(f"cate-idx: {cate_idx}")

In [ ]:
result_list = batcher.handle_message_list(question_list)

In [ ]:
with open(f'pipeline2_step1_{model_name}_response.csv', 'w', newline='', encoding='utf-8') as outfile:
    writer = csv.writer(outfile)
    writer.writerow(['cate-idx', 'l2-name', 'l3-name', 'l4-name', 'prompt', 'response'])

    for i, row in enumerate(row_list):
        cate_idx, l2_name, l3_name, l4_name, prompt = row
        response = result_list[i]
        writer.writerow([cate_idx, l2_name, l3_name, l4_name, prompt, response])